In [3]:
import pandas as pd
import numpy as np
from datetime import datetime

print("=" * 70)
print("STEP 1: LOAD RAW DATA")
print("=" * 70)

# Load the data we just generated
df = pd.read_csv('../data/raw_properties.csv')

print(f"Loaded {len(df)} properties")
print(f"\nFirst few rows:")
print(df.head())
print(f"\nData types:")
print(df.dtypes)
print(f"\nMissing values:")
print(df.isnull().sum())

print("\n" + "=" * 70)
print("STEP 2: DATA CLEANING")
print("=" * 70)

# Convert date to datetime
df['date_listed'] = pd.to_datetime(df['date_listed'])
df['date_scraped'] = pd.to_datetime(df['date_scraped'])

# Remove any duplicates
df = df.drop_duplicates(subset=['title', 'price', 'location'])
print(f"✓ Removed duplicates: {len(df)} properties remain")

# Handle missing values
df['condition'] = df['condition'].fillna('Unknown')
print(f"✓ Filled missing values")

print("\n" + "=" * 70)
print("STEP 3: FEATURE ENGINEERING")
print("=" * 70)

# Calculate days since listed
df['days_listed'] = (pd.to_datetime('today') - df['date_listed']).dt.days
print(f"✓ Created: days_listed (how long property has been listed)")

# Price category
df['price_category'] = pd.cut(df['price'], 
                               bins=[0, 75000, 150000, 250000, 500000, float('inf')],
                               labels=['Budget', 'Standard', 'Premium', 'Luxury', 'Ultra-Luxury'])
print(f"✓ Created: price_category")

# Price per m² (for Land, skip if area is very large)
df['price_per_m2'] = (df['price'] / df['area']).round(2)
print(f"✓ Created: price_per_m2")

# Quality score based on condition
condition_score = {'New': 5, 'Excellent': 4, 'Good': 3, 'Fair': 2, 'Unknown': 1}
df['quality_score'] = df['condition'].map(condition_score)
print(f"✓ Created: quality_score")

# Features score (parking + garden)
df['features_score'] = df['has_parking'].astype(int) + df['has_garden'].astype(int)
print(f"✓ Created: features_score")

# Value score (price_per_m2 ranking - lower is better)
df['value_score'] = 100 - df.groupby('property_type')['price_per_m2'].rank(pct=True) * 100
df['value_score'] = df['value_score'].round(0)
print(f"✓ Created: value_score (investment potential)")

# Market type based on location
def get_market_type(district):
    urban = ['Tunis Centre', 'Sfax Centre', 'Sousse Centre']
    coastal = ['La Marsa', 'Hammamet', 'Nabeul', 'Bizerte', 'Djerba', 'Mahdia', 'Monastir']
    if district in urban:
        return 'Urban'
    elif district in coastal:
        return 'Coastal'
    else:
        return 'Rural'

df['market_type'] = df['district'].apply(get_market_type)
print(f"✓ Created: market_type")

print("\n" + "=" * 70)
print("STEP 4: REMOVE OUTLIERS (Optional)")
print("=" * 70)

original_count = len(df)

# Remove extreme outliers
df = df[(df['price'] >= 20000) & (df['price'] <= 1000000)]
df = df[(df['area'] >= 20) & (df['area'] <= 6000)]

print(f"Removed {original_count - len(df)} outliers")
print(f"Remaining: {len(df)} properties")

print("\n" + "=" * 70)
print("STEP 5: SUMMARY STATISTICS")
print("=" * 70)

print(f"\n💰 PRICE STATISTICS:")
print(f"  Mean: {df['price'].mean():,.0f} TND")
print(f"  Median: {df['price'].median():,.0f} TND")
print(f"  Std Dev: {df['price'].std():,.0f} TND")

print(f"\n📐 AREA STATISTICS:")
print(f"  Mean: {df['area'].mean():.1f} m²")
print(f"  Median: {df['area'].median():.1f} m²")

print(f"\n💵 PRICE PER M² BY TYPE:")
print(df.groupby('property_type')['price_per_m2'].agg(['mean', 'median', 'min', 'max']).round(2))

print(f"\n🏠 PROPERTY DISTRIBUTION:")
print(df['property_type'].value_counts())

print(f"\n📍 LISTINGS BY MARKET TYPE:")
print(df['market_type'].value_counts())

print(f"\n🎯 PRICE CATEGORIES:")
print(df['price_category'].value_counts().sort_index())

print(f"\n📊 TOP 5 BEST VALUE PROPERTIES (Low price_per_m2):")
best_value = df[df['property_type'] != 'Land'].nsmallest(5, 'price_per_m2')[['title', 'price', 'area', 'price_per_m2', 'district']]
print(best_value)

print(f"\n🌟 TOP 5 PREMIUM PROPERTIES:")
premium = df.nlargest(5, 'price')[['title', 'price', 'area', 'bedrooms', 'district']]
print(premium)

print("\n" + "=" * 70)
print("STEP 6: SAVE CLEANED DATA")
print("=" * 70)

# Save cleaned data
df.to_csv('../data/cleaned_properties.csv', index=False)
print(f"✅ Saved cleaned data to: data/cleaned_properties.csv")

# Display final columns
print(f"\n📋 FINAL COLUMNS:")
print(list(df.columns))

print(f"\n✅ DATA CLEANING COMPLETE!")
print(f"\nYou now have {len(df)} clean properties ready for analysis & visualization!")



STEP 1: LOAD RAW DATA
Loaded 200 properties

First few rows:
   id                           title    price  price_per_m2  \
0   1  Terrain 4072m² à Sousse Centre  4733138       1162.36   
1   2       Terrain 2733m² à La Marsa  6567611       2403.08   
2   3         Villa 3P 319m² - Mahdia   871935       2733.34   
3   4      Duplex 4P 93m² - Ben Arous    59841        643.45   
4   5      Villa 6P 184m² - Kasserine   272586       1481.45   

                  location       district  area  bedrooms property_type  \
0  Sousse Centre - Tunisia  Sousse Centre  4072         0          Land   
1       La Marsa - Tunisia       La Marsa  2733         0          Land   
2         Mahdia - Tunisia         Mahdia   319         3         Villa   
3      Ben Arous - Tunisia      Ben Arous    93         4        Duplex   
4      Kasserine - Tunisia      Kasserine   184         6         Villa   

  condition  has_parking  has_garden date_listed date_scraped  
0      Good         True       False  2